# Image circuit experiments with a separate training repository

Run the setup cell below, then run the existing experiments. Cirkit is installed from its `data-driven-initialization` branch; training is loaded from `cirkit-circuits-training`.

The setup clones `https://github.com/GianpieroTulipani/cirkit-circuits-training`. Change the clone URL if you choose a different repository name. On Kaggle, enable Internet. An existing clone is reused without automatically pulling updates.

The experiment outputs and results table below are preserved from the supplied notebook, not recomputed by this update. To reproduce a run, pin both repositories to the corresponding commits and record dependency versions and hardware. W&B is installed by the setup cell. Every experiment below enables `--wandb`, logs at step 0 and every 250 optimizer updates (plus the last step), and fixes the split seed to 42. The setup cell calls `wandb.login()` before starting the experiments; enter your API key through its prompt, not in notebook source. Existing outputs are historical and contain no new W&B curves.

All experiment commands now use k=64 and at most 100 epochs, with early stopping enabled at patience 5 and no step cap. The preserved outputs below refer to older settings and are not results of these commands. New runs log only `val_nll` and `val_nll_relative` against `step`. Each dataset/RG pair goes to `pc_<dataset>_<rg>_eval`; preserved output that mentions `pc_dataset_eval` is historical.

To run one dataset concurrently after setup, use `!python -m scripts.run_learning_curves --dataset mnist --gpus 0 1 --jobs-per-gpu 2` in a separate cell, instead of running the individual experiment cells below. This launches the eight paired comparisons for that dataset. Change `--dataset` to `fashion-mnist` or `cifar10`; use `--dry-run` to preview the commands.


In [ ]:
from pathlib import Path
import os
import subprocess

workspace = Path.cwd()
if Path('/kaggle/working').exists():
    workspace = Path('/kaggle/working')

training_repo = workspace / 'cirkit-circuits-training'
if workspace.name == 'cirkit-circuits-training':
    training_repo = workspace
if not training_repo.exists():
    subprocess.run([
        'git', 'clone',
        'https://github.com/GianpieroTulipani/cirkit-circuits-training.git',
        str(training_repo),
    ], check=True)
os.chdir(training_repo)

%pip install -r requirements.txt

import wandb
wandb.login()


## Experimental results

We compare **data-driven** and **random initialization** across four region graphs (RGs) on MNIST, Fashion-MNIST and CIFAR-10. All configurations use **128 units** (`--k 128`), **CP** inner layers and **seed 42**.

Datasets are listed in rows. Columns are grouped by initialization method, with one column per RG within each group. Each entry reports **test bpd (bits per dimension)**. **Bold** marks the best available RG-initialization combination for each dataset, selected using the saved test bpd values before rounding. Results are taken from the saved cell outputs; an em dash indicates that no test result is available.

<table>
  <thead>
    <tr>
      <th rowspan="2" scope="col">Dataset</th>
      <th colspan="4" scope="colgroup">Data-driven initialization</th>
      <th colspan="4" scope="colgroup">Random initialization</th>
    </tr>
    <tr>
      <th scope="col">Quad-Tree-2</th>
      <th scope="col">Quad-Tree-4</th>
      <th scope="col">Quad-Graph</th>
      <th scope="col">Chow-Liu Tree</th>
      <th scope="col">Quad-Tree-2</th>
      <th scope="col">Quad-Tree-4</th>
      <th scope="col">Quad-Graph</th>
      <th scope="col">Chow-Liu Tree</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th scope="row">MNIST</th>
      <td>1.1644</td>
      <td>1.1734</td>
      <td><strong>1.1620</strong></td>
      <td>1.1847</td>
      <td>1.1788</td>
      <td>1.1871</td>
      <td>1.1784</td>
      <td>1.1909</td>
    </tr>
    <tr>
      <th scope="row">Fashion-MNIST</th>
      <td>3.3841</td>
      <td>3.4051</td>
      <td><strong>3.3222</strong></td>
      <td>3.3477</td>
      <td>3.3840</td>
      <td>3.3925</td>
      <td>3.3251</td>
      <td>3.3288</td>
    </tr>
    <tr>
      <th scope="row">CIFAR-10</th>
      <td>4.8919</td>
      <td>4.8999</td>
      <td><strong>4.8157</strong></td>
      <td>&mdash;</td>
      <td>4.8805</td>
      <td>4.9174</td>
      <td>4.8851</td>
      <td>&mdash;</td>
    </tr>
  </tbody>
</table>

For CIFAR-10, results refer to images transformed with `--ycc lossy`. No test outputs are saved for either Chow-Liu Tree initialization on CIFAR-10.

# Dataset: MNIST

Grayscale 28 × 28 images, without color transformation; input sharing set to `none`.

All experiments use **128 units**, **CP** inner layers and **seed 42**.

## Region Graph: Quad-Tree-2

### Data Initialization

In [8]:
!python -m scripts.evaluate_image_circuit \
    --rg quad-tree-2 --dataset mnist \
    --input-sharing none --ycc none \
    --activation none --inner-layer cp --use-mixing-weights \
    --use-estimated-weights --alpha 5.0 --noise-scale 2.0 \
    --k 64 --seed 42 \
    --wandb --validation-steps 250 --split-seed 42 --max-epochs 100 --patience 5 \
    --save-path mnist_quad-tree-2_data-init_k64_seed42.pt

  dataset: mnist
  root: datasets
  image_shape: None
  ycc: none
  input_sharing: none
  rg: quad-tree-2
  num_bins: None
  mi_chunk_size: None
  inner_layer: cp
  k: 128
  activation: none
  weights_init: uniform
  lr: 0.01
  T_0: 1
  eta_min: 0.0001
  weight_decay: 0.0
  batch_size: 256
  max_epochs: 200
  validation_steps: 250
  patience: 5
  min_delta: 0.0
  valid_split: 0.05
  use_scheduler: False
  alpha: 5.0
  noise_scale: 2.0
  use_mixing_weights: True
  use_estimated_weights: True
  torch_compile: False
  compile_backend: inductor
  compile_mode: default
  compile_fullgraph: False
  wandb: False
  project: pc_dataset_eval
  seed: 42
  save_path: mnist_quad-tree-2_data-init_k64_seed42.pt
  device: cuda
Loaded mnist: train=(57000, 784), val=(3000, 784), test=(10000, 784)
Using image_shape=(1, 28, 28), ycc=none, input_sharing=none
Using 57000 samples to build/estimate the circuit
Circuit built with 3134 layers
Number of parameters: 51347584
Starting training for a maximum of 446

### Random Initialization

In [5]:
!python -m scripts.evaluate_image_circuit \
    --rg quad-tree-2 --dataset mnist \
    --input-sharing none --ycc none \
    --activation none --inner-layer cp --use-mixing-weights \
    --k 64 --seed 42 \
    --wandb --validation-steps 250 --split-seed 42 --max-epochs 100 --patience 5 \
    --save-path mnist_quad-tree-2_random-init_k64_seed42.pt

  dataset: mnist
  root: datasets
  image_shape: None
  ycc: none
  input_sharing: none
  rg: quad-tree-2
  num_bins: None
  mi_chunk_size: None
  inner_layer: cp
  k: 128
  activation: none
  weights_init: uniform
  lr: 0.01
  T_0: 1
  eta_min: 0.0001
  weight_decay: 0.0
  batch_size: 256
  max_epochs: 200
  validation_steps: 250
  patience: 5
  min_delta: 0.0
  valid_split: 0.05
  use_scheduler: False
  alpha: 5.0
  noise_scale: 2.0
  use_mixing_weights: True
  use_estimated_weights: False
  torch_compile: False
  compile_backend: inductor
  compile_mode: default
  compile_fullgraph: False
  wandb: False
  project: pc_dataset_eval
  seed: 42
  save_path: mnist_quad-tree-2_random-init_k64_seed42.pt
  device: cuda
Loaded mnist: train=(57000, 784), val=(3000, 784), test=(10000, 784)
Using image_shape=(1, 28, 28), ycc=none, input_sharing=none
Using 57000 samples to build/estimate the circuit
Circuit built with 3134 layers
Number of parameters: 51347584
Starting training for a maximum of 

## Region Graph: Quad-Tree-4

### Data Initialization

In [2]:
!python -m scripts.evaluate_image_circuit \
    --rg quad-tree-4 --dataset mnist \
    --input-sharing none --ycc none \
    --activation none --inner-layer cp --use-mixing-weights \
    --use-estimated-weights --alpha 5.0 --noise-scale 2.0 \
    --k 64 --seed 42 \
    --wandb --validation-steps 250 --split-seed 42 --max-epochs 100 --patience 5 \
    --save-path mnist_quad-tree-4_data-init_k64_seed42.pt

  dataset: mnist
  root: datasets
  image_shape: None
  ycc: none
  input_sharing: none
  rg: quad-tree-4
  num_bins: None
  mi_chunk_size: None
  inner_layer: cp
  k: 128
  activation: none
  weights_init: uniform
  lr: 0.01
  T_0: 1
  eta_min: 0.0001
  weight_decay: 0.0
  batch_size: 256
  max_epochs: 200
  validation_steps: 250
  patience: 5
  min_delta: 0.0
  valid_split: 0.05
  use_scheduler: False
  alpha: 5.0
  noise_scale: 2.0
  use_mixing_weights: True
  use_estimated_weights: True
  torch_compile: False
  compile_backend: inductor
  compile_mode: default
  compile_fullgraph: False
  wandb: False
  project: pc_dataset_eval
  seed: 42
  save_path: mnist_quad-tree-4_data-init_k64_seed42.pt
  device: cuda
100%|██████████████████████████████████████| 9.91M/9.91M [00:01<00:00, 5.83MB/s]
100%|███████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 157kB/s]
100%|██████████████████████████████████████| 1.65M/1.65M [00:01<00:00, 1.13MB/s]
100%|███████████████████████████████

### Random Initialization

In [3]:
!python -m scripts.evaluate_image_circuit \
    --rg quad-tree-4 --dataset mnist \
    --input-sharing none --ycc none \
    --activation none --inner-layer cp --use-mixing-weights \
    --k 64 --seed 42 \
    --wandb --validation-steps 250 --split-seed 42 --max-epochs 100 --patience 5 \
    --save-path mnist_quad-tree-4_random-init_k64_seed42.pt

  dataset: mnist
  root: datasets
  image_shape: None
  ycc: none
  input_sharing: none
  rg: quad-tree-4
  num_bins: None
  mi_chunk_size: None
  inner_layer: cp
  k: 128
  activation: none
  weights_init: uniform
  lr: 0.01
  T_0: 1
  eta_min: 0.0001
  weight_decay: 0.0
  batch_size: 256
  max_epochs: 200
  validation_steps: 250
  patience: 5
  min_delta: 0.0
  valid_split: 0.05
  use_scheduler: False
  alpha: 5.0
  noise_scale: 2.0
  use_mixing_weights: True
  use_estimated_weights: False
  torch_compile: False
  compile_backend: inductor
  compile_mode: default
  compile_fullgraph: False
  wandb: False
  project: pc_dataset_eval
  seed: 42
  save_path: mnist_quad-tree-4_random-init_k64_seed42.pt
  device: cuda
Loaded mnist: train=(57000, 784), val=(3000, 784), test=(10000, 784)
Using image_shape=(1, 28, 28), ycc=none, input_sharing=none
Using 57000 samples to build/estimate the circuit
Circuit built with 2098 layers
Number of parameters: 42860672
Starting training for a maximum of 

## Region Graph: Quad-Graph

### Data Initialization

In [4]:
!python -m scripts.evaluate_image_circuit \
    --rg quad-graph --dataset mnist \
    --input-sharing none --ycc none \
    --activation none --inner-layer cp --use-mixing-weights \
    --use-estimated-weights --alpha 5.0 --noise-scale 2.0 \
    --k 64 --seed 42 \
    --wandb --validation-steps 250 --split-seed 42 --max-epochs 100 --patience 5 \
    --save-path mnist_quad-graph_data-init_k64_seed42.pt

  dataset: mnist
  root: datasets
  image_shape: None
  ycc: none
  input_sharing: none
  rg: quad-graph
  num_bins: None
  mi_chunk_size: None
  inner_layer: cp
  k: 128
  activation: none
  weights_init: uniform
  lr: 0.01
  T_0: 1
  eta_min: 0.0001
  weight_decay: 0.0
  batch_size: 256
  max_epochs: 200
  validation_steps: 250
  patience: 5
  min_delta: 0.0
  valid_split: 0.05
  use_scheduler: False
  alpha: 5.0
  noise_scale: 2.0
  use_mixing_weights: True
  use_estimated_weights: True
  torch_compile: False
  compile_backend: inductor
  compile_mode: default
  compile_fullgraph: False
  wandb: False
  project: pc_dataset_eval
  seed: 42
  save_path: mnist_quad-graph_data-init_k64_seed42.pt
  device: cuda
Loaded mnist: train=(57000, 784), val=(3000, 784), test=(10000, 784)
Using image_shape=(1, 28, 28), ycc=none, input_sharing=none
Using 57000 samples to build/estimate the circuit
Circuit built with 5725 layers
Number of parameters: 76874498
Starting training for a maximum of 44600

### Random Initialization

In [5]:
!python -m scripts.evaluate_image_circuit \
    --rg quad-graph --dataset mnist \
    --input-sharing none --ycc none \
    --activation none --inner-layer cp --use-mixing-weights \
    --k 64 --seed 42 \
    --wandb --validation-steps 250 --split-seed 42 --max-epochs 100 --patience 5 \
    --save-path mnist_quad-graph_random-init_k64_seed42.pt

  dataset: mnist
  root: datasets
  image_shape: None
  ycc: none
  input_sharing: none
  rg: quad-graph
  num_bins: None
  mi_chunk_size: None
  inner_layer: cp
  k: 128
  activation: none
  weights_init: uniform
  lr: 0.01
  T_0: 1
  eta_min: 0.0001
  weight_decay: 0.0
  batch_size: 256
  max_epochs: 200
  validation_steps: 250
  patience: 5
  min_delta: 0.0
  valid_split: 0.05
  use_scheduler: False
  alpha: 5.0
  noise_scale: 2.0
  use_mixing_weights: True
  use_estimated_weights: False
  torch_compile: False
  compile_backend: inductor
  compile_mode: default
  compile_fullgraph: False
  wandb: False
  project: pc_dataset_eval
  seed: 42
  save_path: mnist_quad-graph_random-init_k64_seed42.pt
  device: cuda
Loaded mnist: train=(57000, 784), val=(3000, 784), test=(10000, 784)
Using image_shape=(1, 28, 28), ycc=none, input_sharing=none
Using 57000 samples to build/estimate the circuit
Circuit built with 5725 layers
Number of parameters: 76874498
Starting training for a maximum of 44

## Region Graph: Chow-Liu Tree

The structure is learned from the training set for both initialization methods. `--num-bins 8` only affects graph construction: the input distributions retain 256 levels. `--mi-chunk-size 128` limits the number of samples processed together when computing mutual information.

### Data Initialization

In [8]:
!python -m scripts.evaluate_image_circuit \
    --rg chow-liu-tree --dataset mnist \
    --input-sharing none --ycc none \
    --activation none --inner-layer cp --use-mixing-weights \
    --use-estimated-weights --alpha 5.0 --noise-scale 2.0 \
    --k 64 --seed 42 \
    --num-bins 8 --mi-chunk-size 128 \
    --wandb --validation-steps 250 --split-seed 42 --max-epochs 100 --patience 5 \
    --save-path mnist_chow-liu-tree_data-init_k64_seed42.pt

  dataset: mnist
  root: datasets
  image_shape: None
  ycc: none
  input_sharing: none
  rg: chow-liu-tree
  num_bins: 8
  mi_chunk_size: 128
  inner_layer: cp
  k: 128
  activation: none
  weights_init: uniform
  lr: 0.01
  T_0: 1
  eta_min: 0.0001
  weight_decay: 0.0
  batch_size: 256
  max_epochs: 200
  validation_steps: 250
  patience: 5
  min_delta: 0.0
  valid_split: 0.05
  use_scheduler: False
  alpha: 5.0
  noise_scale: 2.0
  use_mixing_weights: True
  use_estimated_weights: True
  torch_compile: False
  compile_backend: inductor
  compile_mode: default
  compile_fullgraph: False
  wandb: False
  project: pc_dataset_eval
  seed: 42
  save_path: mnist_chow-liu-tree_data-init_k64_seed42.pt
  device: cuda
Loaded mnist: train=(57000, 784), val=(3000, 784), test=(10000, 784)
Using image_shape=(1, 28, 28), ycc=none, input_sharing=none
Using 57000 samples to build/estimate the circuit
Circuit built with 2752 layers
Number of parameters: 48218240
Starting training for a maximum of 446

### Random Initialization

In [9]:
!python -m scripts.evaluate_image_circuit \
    --rg chow-liu-tree --dataset mnist \
    --input-sharing none --ycc none \
    --activation none --inner-layer cp --use-mixing-weights \
    --k 64 --seed 42 \
    --num-bins 8 --mi-chunk-size 128 \
    --wandb --validation-steps 250 --split-seed 42 --max-epochs 100 --patience 5 \
    --save-path mnist_chow-liu-tree_random-init_k64_seed42.pt

  dataset: mnist
  root: datasets
  image_shape: None
  ycc: none
  input_sharing: none
  rg: chow-liu-tree
  num_bins: 8
  mi_chunk_size: 128
  inner_layer: cp
  k: 128
  activation: none
  weights_init: uniform
  lr: 0.01
  T_0: 1
  eta_min: 0.0001
  weight_decay: 0.0
  batch_size: 256
  max_epochs: 200
  validation_steps: 250
  patience: 5
  min_delta: 0.0
  valid_split: 0.05
  use_scheduler: False
  alpha: 5.0
  noise_scale: 2.0
  use_mixing_weights: True
  use_estimated_weights: False
  torch_compile: False
  compile_backend: inductor
  compile_mode: default
  compile_fullgraph: False
  wandb: False
  project: pc_dataset_eval
  seed: 42
  save_path: mnist_chow-liu-tree_random-init_k64_seed42.pt
  device: cuda
Loaded mnist: train=(57000, 784), val=(3000, 784), test=(10000, 784)
Using image_shape=(1, 28, 28), ycc=none, input_sharing=none
Using 57000 samples to build/estimate the circuit
Circuit built with 2752 layers
Number of parameters: 48218240
Starting training for a maximum of 

# Dataset: Fashion-MNIST

Grayscale 28 × 28 images, without color transformation; input sharing set to `none`.

All experiments use **128 units**, **CP** inner layers and **seed 42**.

## Region Graph: Quad-Tree-2

### Data Initialization

In [10]:
!python -m scripts.evaluate_image_circuit \
    --rg quad-tree-2 --dataset fashion-mnist \
    --input-sharing none --ycc none \
    --activation none --inner-layer cp --use-mixing-weights \
    --use-estimated-weights --alpha 5.0 --noise-scale 2.0 \
    --k 64 --seed 42 \
    --wandb --validation-steps 250 --split-seed 42 --max-epochs 100 --patience 5 \
    --save-path fashion-mnist_quad-tree-2_data-init_k64_seed42.pt

  dataset: fashion-mnist
  root: datasets
  image_shape: None
  ycc: none
  input_sharing: none
  rg: quad-tree-2
  num_bins: None
  mi_chunk_size: None
  inner_layer: cp
  k: 128
  activation: none
  weights_init: uniform
  lr: 0.01
  T_0: 1
  eta_min: 0.0001
  weight_decay: 0.0
  batch_size: 256
  max_epochs: 200
  validation_steps: 250
  patience: 5
  min_delta: 0.0
  valid_split: 0.05
  use_scheduler: False
  alpha: 5.0
  noise_scale: 2.0
  use_mixing_weights: True
  use_estimated_weights: True
  torch_compile: False
  compile_backend: inductor
  compile_mode: default
  compile_fullgraph: False
  wandb: False
  project: pc_dataset_eval
  seed: 42
  save_path: fashion-mnist_quad-tree-2_data-init_k128_seed42.pt
  device: cuda
100%|██████████████████████████████████████| 26.4M/26.4M [00:02<00:00, 9.49MB/s]
100%|███████████████████████████████████████| 29.5k/29.5k [00:00<00:00, 143kB/s]
100%|██████████████████████████████████████| 4.42M/4.42M [00:01<00:00, 2.66MB/s]
100%|██████████████

### Random Initialization

In [11]:
!python -m scripts.evaluate_image_circuit \
    --rg quad-tree-2 --dataset fashion-mnist \
    --input-sharing none --ycc none \
    --activation none --inner-layer cp --use-mixing-weights \
    --k 64 --seed 42 \
    --wandb --validation-steps 250 --split-seed 42 --max-epochs 100 --patience 5 \
    --save-path fashion-mnist_quad-tree-2_random-init_k64_seed42.pt

  dataset: fashion-mnist
  root: datasets
  image_shape: None
  ycc: none
  input_sharing: none
  rg: quad-tree-2
  num_bins: None
  mi_chunk_size: None
  inner_layer: cp
  k: 128
  activation: none
  weights_init: uniform
  lr: 0.01
  T_0: 1
  eta_min: 0.0001
  weight_decay: 0.0
  batch_size: 256
  max_epochs: 200
  validation_steps: 250
  patience: 5
  min_delta: 0.0
  valid_split: 0.05
  use_scheduler: False
  alpha: 5.0
  noise_scale: 2.0
  use_mixing_weights: True
  use_estimated_weights: False
  torch_compile: False
  compile_backend: inductor
  compile_mode: default
  compile_fullgraph: False
  wandb: False
  project: pc_dataset_eval
  seed: 42
  save_path: fashion-mnist_quad-tree-2_random-init_k128_seed42.pt
  device: cuda
Loaded fashion-mnist: train=(57000, 784), val=(3000, 784), test=(10000, 784)
Using image_shape=(1, 28, 28), ycc=none, input_sharing=none
Using 57000 samples to build/estimate the circuit
Circuit built with 3134 layers
Number of parameters: 51347584
Starting t

## Region Graph: Quad-Tree-4

### Data Initialization

In [12]:
!python -m scripts.evaluate_image_circuit \
    --rg quad-tree-4 --dataset fashion-mnist \
    --input-sharing none --ycc none \
    --activation none --inner-layer cp --use-mixing-weights \
    --use-estimated-weights --alpha 5.0 --noise-scale 2.0 \
    --k 64 --seed 42 \
    --wandb --validation-steps 250 --split-seed 42 --max-epochs 100 --patience 5 \
    --save-path fashion-mnist_quad-tree-4_data-init_k64_seed42.pt

  dataset: fashion-mnist
  root: datasets
  image_shape: None
  ycc: none
  input_sharing: none
  rg: quad-tree-4
  num_bins: None
  mi_chunk_size: None
  inner_layer: cp
  k: 128
  activation: none
  weights_init: uniform
  lr: 0.01
  T_0: 1
  eta_min: 0.0001
  weight_decay: 0.0
  batch_size: 256
  max_epochs: 200
  validation_steps: 250
  patience: 5
  min_delta: 0.0
  valid_split: 0.05
  use_scheduler: False
  alpha: 5.0
  noise_scale: 2.0
  use_mixing_weights: True
  use_estimated_weights: True
  torch_compile: False
  compile_backend: inductor
  compile_mode: default
  compile_fullgraph: False
  wandb: False
  project: pc_dataset_eval
  seed: 42
  save_path: fashion-mnist_quad-tree-4_data-init_k128_seed42.pt
  device: cuda
Loaded fashion-mnist: train=(57000, 784), val=(3000, 784), test=(10000, 784)
Using image_shape=(1, 28, 28), ycc=none, input_sharing=none
Using 57000 samples to build/estimate the circuit
Circuit built with 2098 layers
Number of parameters: 42860672
Starting trai

### Random Initialization

In [13]:
!python -m scripts.evaluate_image_circuit \
    --rg quad-tree-4 --dataset fashion-mnist \
    --input-sharing none --ycc none \
    --activation none --inner-layer cp --use-mixing-weights \
    --k 64 --seed 42 \
    --wandb --validation-steps 250 --split-seed 42 --max-epochs 100 --patience 5 \
    --save-path fashion-mnist_quad-tree-4_random-init_k64_seed42.pt

  dataset: fashion-mnist
  root: datasets
  image_shape: None
  ycc: none
  input_sharing: none
  rg: quad-tree-4
  num_bins: None
  mi_chunk_size: None
  inner_layer: cp
  k: 128
  activation: none
  weights_init: uniform
  lr: 0.01
  T_0: 1
  eta_min: 0.0001
  weight_decay: 0.0
  batch_size: 256
  max_epochs: 200
  validation_steps: 250
  patience: 5
  min_delta: 0.0
  valid_split: 0.05
  use_scheduler: False
  alpha: 5.0
  noise_scale: 2.0
  use_mixing_weights: True
  use_estimated_weights: False
  torch_compile: False
  compile_backend: inductor
  compile_mode: default
  compile_fullgraph: False
  wandb: False
  project: pc_dataset_eval
  seed: 42
  save_path: fashion-mnist_quad-tree-4_random-init_k128_seed42.pt
  device: cuda
Loaded fashion-mnist: train=(57000, 784), val=(3000, 784), test=(10000, 784)
Using image_shape=(1, 28, 28), ycc=none, input_sharing=none
Using 57000 samples to build/estimate the circuit
Circuit built with 2098 layers
Number of parameters: 42860672
Starting t

## Region Graph: Quad-Graph

### Data Initialization

In [14]:
!python -m scripts.evaluate_image_circuit \
    --rg quad-graph --dataset fashion-mnist \
    --input-sharing none --ycc none \
    --activation none --inner-layer cp --use-mixing-weights \
    --use-estimated-weights --alpha 5.0 --noise-scale 2.0 \
    --k 64 --seed 42 \
    --wandb --validation-steps 250 --split-seed 42 --max-epochs 100 --patience 5 \
    --save-path fashion-mnist_quad-graph_data-init_k64_seed42.pt

  dataset: fashion-mnist
  root: datasets
  image_shape: None
  ycc: none
  input_sharing: none
  rg: quad-graph
  num_bins: None
  mi_chunk_size: None
  inner_layer: cp
  k: 128
  activation: none
  weights_init: uniform
  lr: 0.01
  T_0: 1
  eta_min: 0.0001
  weight_decay: 0.0
  batch_size: 256
  max_epochs: 200
  validation_steps: 250
  patience: 5
  min_delta: 0.0
  valid_split: 0.05
  use_scheduler: False
  alpha: 5.0
  noise_scale: 2.0
  use_mixing_weights: True
  use_estimated_weights: True
  torch_compile: False
  compile_backend: inductor
  compile_mode: default
  compile_fullgraph: False
  wandb: False
  project: pc_dataset_eval
  seed: 42
  save_path: fashion-mnist_quad-graph_data-init_k128_seed42.pt
  device: cuda
Loaded fashion-mnist: train=(57000, 784), val=(3000, 784), test=(10000, 784)
Using image_shape=(1, 28, 28), ycc=none, input_sharing=none
Using 57000 samples to build/estimate the circuit
Circuit built with 5725 layers
Number of parameters: 76874498
Starting traini

### Random Initialization

In [15]:
!python -m scripts.evaluate_image_circuit \
    --rg quad-graph --dataset fashion-mnist \
    --input-sharing none --ycc none \
    --activation none --inner-layer cp --use-mixing-weights \
    --k 64 --seed 42 \
    --wandb --validation-steps 250 --split-seed 42 --max-epochs 100 --patience 5 \
    --save-path fashion-mnist_quad-graph_random-init_k64_seed42.pt

  dataset: fashion-mnist
  root: datasets
  image_shape: None
  ycc: none
  input_sharing: none
  rg: quad-graph
  num_bins: None
  mi_chunk_size: None
  inner_layer: cp
  k: 128
  activation: none
  weights_init: uniform
  lr: 0.01
  T_0: 1
  eta_min: 0.0001
  weight_decay: 0.0
  batch_size: 256
  max_epochs: 200
  validation_steps: 250
  patience: 5
  min_delta: 0.0
  valid_split: 0.05
  use_scheduler: False
  alpha: 5.0
  noise_scale: 2.0
  use_mixing_weights: True
  use_estimated_weights: False
  torch_compile: False
  compile_backend: inductor
  compile_mode: default
  compile_fullgraph: False
  wandb: False
  project: pc_dataset_eval
  seed: 42
  save_path: fashion-mnist_quad-graph_random-init_k128_seed42.pt
  device: cuda
Loaded fashion-mnist: train=(57000, 784), val=(3000, 784), test=(10000, 784)
Using image_shape=(1, 28, 28), ycc=none, input_sharing=none
Using 57000 samples to build/estimate the circuit
Circuit built with 5725 layers
Number of parameters: 76874498
Starting tra

## Region Graph: Chow-Liu Tree

The structure is learned from the training set for both initialization methods. `--num-bins 8` only affects graph construction: the input distributions retain 256 levels. `--mi-chunk-size 128` limits the number of samples processed together when computing mutual information.

### Data Initialization

In [2]:
!python -m scripts.evaluate_image_circuit \
    --rg chow-liu-tree --dataset fashion-mnist \
    --input-sharing none --ycc none \
    --activation none --inner-layer cp --use-mixing-weights \
    --use-estimated-weights --alpha 5.0 --noise-scale 2.0 \
    --k 64 --seed 42 \
    --num-bins 8 --mi-chunk-size 128 \
    --wandb --validation-steps 250 --split-seed 42 --max-epochs 100 --patience 5 \
    --save-path fashion-mnist_chow-liu-tree_data-init_k64_seed42.pt

  dataset: fashion-mnist
  root: datasets
  image_shape: None
  ycc: none
  input_sharing: none
  rg: chow-liu-tree
  num_bins: 8
  mi_chunk_size: 128
  inner_layer: cp
  k: 128
  activation: none
  weights_init: uniform
  lr: 0.01
  T_0: 1
  eta_min: 0.0001
  weight_decay: 0.0
  batch_size: 256
  max_epochs: 200
  validation_steps: 250
  patience: 5
  min_delta: 0.0
  valid_split: 0.05
  use_scheduler: False
  alpha: 5.0
  noise_scale: 2.0
  use_mixing_weights: True
  use_estimated_weights: True
  torch_compile: False
  compile_backend: inductor
  compile_mode: default
  compile_fullgraph: False
  wandb: False
  project: pc_dataset_eval
  seed: 42
  save_path: fashion-mnist_chow-liu-tree_data-init_k128_seed42.pt
  device: cuda
100%|██████████████████████████████████████| 26.4M/26.4M [00:01<00:00, 19.0MB/s]
100%|███████████████████████████████████████| 29.5k/29.5k [00:00<00:00, 301kB/s]
100%|██████████████████████████████████████| 4.42M/4.42M [00:00<00:00, 5.64MB/s]
100%|██████████████

### Random Initialization

In [3]:
!python -m scripts.evaluate_image_circuit \
    --rg chow-liu-tree --dataset fashion-mnist \
    --input-sharing none --ycc none \
    --activation none --inner-layer cp --use-mixing-weights \
    --k 64 --seed 42 \
    --num-bins 8 --mi-chunk-size 128 \
    --wandb --validation-steps 250 --split-seed 42 --max-epochs 100 --patience 5 \
    --save-path fashion-mnist_chow-liu-tree_random-init_k64_seed42.pt

  dataset: fashion-mnist
  root: datasets
  image_shape: None
  ycc: none
  input_sharing: none
  rg: chow-liu-tree
  num_bins: 8
  mi_chunk_size: 128
  inner_layer: cp
  k: 128
  activation: none
  weights_init: uniform
  lr: 0.01
  T_0: 1
  eta_min: 0.0001
  weight_decay: 0.0
  batch_size: 256
  max_epochs: 200
  validation_steps: 250
  patience: 5
  min_delta: 0.0
  valid_split: 0.05
  use_scheduler: False
  alpha: 5.0
  noise_scale: 2.0
  use_mixing_weights: True
  use_estimated_weights: False
  torch_compile: False
  compile_backend: inductor
  compile_mode: default
  compile_fullgraph: False
  wandb: False
  project: pc_dataset_eval
  seed: 42
  save_path: fashion-mnist_chow-liu-tree_random-init_k128_seed42.pt
  device: cuda
Loaded fashion-mnist: train=(57000, 784), val=(3000, 784), test=(10000, 784)
Using image_shape=(1, 28, 28), ycc=none, input_sharing=none
Using 57000 samples to build/estimate the circuit
Circuit built with 2886 layers
Number of parameters: 49315968
Starting t

# Dataset: CIFAR-10

RGB 32 x 32 images, transformed to YCoCg with `--ycc lossy`, as in the original notebook. This transformation entails information loss. Input sharing is set to `full` for the three quad RGs and `none` for Chow-Liu, which does not support multichannel input sharing in this script.

All experiments use **128 units**, **CP** inner layers and **seed 42**. Each experiment saves a separate checkpoint.

## Region Graph: Quad-Tree-2

### Data Initialization

In [4]:
!python -m scripts.evaluate_image_circuit \
    --rg quad-tree-2 --dataset cifar10 \
    --input-sharing full --ycc lossy \
    --activation none --inner-layer cp --use-mixing-weights \
    --use-estimated-weights --alpha 5.0 --noise-scale 2.0 \
    --k 64 --seed 42 \
    --wandb --validation-steps 250 --split-seed 42 --max-epochs 100 --patience 5 \
    --save-path cifar10_quad-tree-2_data-init_k64_seed42.pt

  dataset: cifar10
  root: datasets
  image_shape: None
  ycc: lossy
  input_sharing: full
  rg: quad-tree-2
  num_bins: None
  mi_chunk_size: None
  inner_layer: cp
  k: 128
  activation: none
  weights_init: uniform
  lr: 0.01
  T_0: 1
  eta_min: 0.0001
  weight_decay: 0.0
  batch_size: 256
  max_epochs: 200
  validation_steps: 250
  patience: 5
  min_delta: 0.0
  valid_split: 0.05
  use_scheduler: False
  alpha: 5.0
  noise_scale: 2.0
  use_mixing_weights: True
  use_estimated_weights: True
  torch_compile: False
  compile_backend: inductor
  compile_mode: default
  compile_fullgraph: False
  wandb: False
  project: pc_dataset_eval
  seed: 42
  save_path: cifar10_quad-tree-2_data-init_k128_seed42.pt
  device: cuda
100%|█████████████████████████████████████████| 170M/170M [11:31<00:00, 247kB/s]
Loaded cifar10: train=(47500, 3072), val=(2500, 3072), test=(10000, 3072)
Using image_shape=(3, 32, 32), ycc=lossy, input_sharing=full
Using 47500 samples to build/estimate the circuit
Circuit

### Random Initialization

In [2]:
!python -m scripts.evaluate_image_circuit \
    --rg quad-tree-2 --dataset cifar10 \
    --input-sharing full --ycc lossy \
    --activation none --inner-layer cp --use-mixing-weights \
    --k 64 --seed 42 \
    --wandb --validation-steps 250 --split-seed 42 --max-epochs 100 --patience 5 \
    --save-path cifar10_quad-tree-2_random-init_k64_seed42.pt

  dataset: cifar10
  root: datasets
  image_shape: None
  ycc: lossy
  input_sharing: full
  rg: quad-tree-2
  num_bins: None
  mi_chunk_size: None
  inner_layer: cp
  k: 128
  activation: none
  weights_init: uniform
  lr: 0.01
  T_0: 1
  eta_min: 0.0001
  weight_decay: 0.0
  batch_size: 256
  max_epochs: 200
  validation_steps: 250
  patience: 5
  min_delta: 0.0
  valid_split: 0.05
  use_scheduler: False
  alpha: 5.0
  noise_scale: 2.0
  use_mixing_weights: True
  use_estimated_weights: False
  torch_compile: False
  compile_backend: inductor
  compile_mode: default
  compile_fullgraph: False
  wandb: False
  project: pc_dataset_eval
  seed: 42
  save_path: cifar10_quad-tree-2_random-init_k128_seed42.pt
  device: cuda
100%|████████████████████████████████████████| 170M/170M [01:20<00:00, 2.11MB/s]
Loaded cifar10: train=(47500, 3072), val=(2500, 3072), test=(10000, 3072)
Using image_shape=(3, 32, 32), ycc=lossy, input_sharing=full
Using 47500 samples to build/estimate the circuit
Circ

## Region Graph: Quad-Tree-4

### Data Initialization

In [3]:
!python -m scripts.evaluate_image_circuit \
    --rg quad-tree-4 --dataset cifar10 \
    --input-sharing full --ycc lossy \
    --activation none --inner-layer cp --use-mixing-weights \
    --use-estimated-weights --alpha 5.0 --noise-scale 2.0 \
    --k 64 --seed 42 \
    --wandb --validation-steps 250 --split-seed 42 --max-epochs 100 --patience 5 \
    --save-path cifar10_quad-tree-4_data-init_k64_seed42.pt

  dataset: cifar10
  root: datasets
  image_shape: None
  ycc: lossy
  input_sharing: full
  rg: quad-tree-4
  num_bins: None
  mi_chunk_size: None
  inner_layer: cp
  k: 128
  activation: none
  weights_init: uniform
  lr: 0.01
  T_0: 1
  eta_min: 0.0001
  weight_decay: 0.0
  batch_size: 256
  max_epochs: 200
  validation_steps: 250
  patience: 5
  min_delta: 0.0
  valid_split: 0.05
  use_scheduler: False
  alpha: 5.0
  noise_scale: 2.0
  use_mixing_weights: True
  use_estimated_weights: True
  torch_compile: False
  compile_backend: inductor
  compile_mode: default
  compile_fullgraph: False
  wandb: False
  project: pc_dataset_eval
  seed: 42
  save_path: cifar10_quad-tree-4_data-init_k128_seed42.pt
  device: cuda
Loaded cifar10: train=(47500, 3072), val=(2500, 3072), test=(10000, 3072)
Using image_shape=(3, 32, 32), ycc=lossy, input_sharing=full
Using 47500 samples to build/estimate the circuit
Circuit built with 2730 layers
Number of parameters: 22446208
Starting training for a ma

### Random Initialization

In [4]:
!python -m scripts.evaluate_image_circuit \
    --rg quad-tree-4 --dataset cifar10 \
    --input-sharing full --ycc lossy \
    --activation none --inner-layer cp --use-mixing-weights \
    --k 64 --seed 42 \
    --wandb --validation-steps 250 --split-seed 42 --max-epochs 100 --patience 5 \
    --save-path cifar10_quad-tree-4_random-init_k64_seed42.pt

  dataset: cifar10
  root: datasets
  image_shape: None
  ycc: lossy
  input_sharing: full
  rg: quad-tree-4
  num_bins: None
  mi_chunk_size: None
  inner_layer: cp
  k: 128
  activation: none
  weights_init: uniform
  lr: 0.01
  T_0: 1
  eta_min: 0.0001
  weight_decay: 0.0
  batch_size: 256
  max_epochs: 200
  validation_steps: 250
  patience: 5
  min_delta: 0.0
  valid_split: 0.05
  use_scheduler: False
  alpha: 5.0
  noise_scale: 2.0
  use_mixing_weights: True
  use_estimated_weights: False
  torch_compile: False
  compile_backend: inductor
  compile_mode: default
  compile_fullgraph: False
  wandb: False
  project: pc_dataset_eval
  seed: 42
  save_path: cifar10_quad-tree-4_random-init_k128_seed42.pt
  device: cuda
Loaded cifar10: train=(47500, 3072), val=(2500, 3072), test=(10000, 3072)
Using image_shape=(3, 32, 32), ycc=lossy, input_sharing=full
Using 47500 samples to build/estimate the circuit
Circuit built with 2730 layers
Number of parameters: 22446208
Starting training for a

## Region Graph: Quad-Graph

### Data Initialization

In [5]:
!python -m scripts.evaluate_image_circuit \
    --rg quad-graph --dataset cifar10 \
    --input-sharing full --ycc lossy \
    --activation none --inner-layer cp --use-mixing-weights \
    --use-estimated-weights --alpha 5.0 --noise-scale 2.0 \
    --k 64 --seed 42 \
    --wandb --validation-steps 250 --split-seed 42 --max-epochs 100 --patience 5 \
    --save-path cifar10_quad-graph_data-init_k64_seed42.pt

  dataset: cifar10
  root: datasets
  image_shape: None
  ycc: lossy
  input_sharing: full
  rg: quad-graph
  num_bins: None
  mi_chunk_size: None
  inner_layer: cp
  k: 128
  activation: none
  weights_init: uniform
  lr: 0.01
  T_0: 1
  eta_min: 0.0001
  weight_decay: 0.0
  batch_size: 256
  max_epochs: 200
  validation_steps: 250
  patience: 5
  min_delta: 0.0
  valid_split: 0.05
  use_scheduler: False
  alpha: 5.0
  noise_scale: 2.0
  use_mixing_weights: True
  use_estimated_weights: True
  torch_compile: False
  compile_backend: inductor
  compile_mode: default
  compile_fullgraph: False
  wandb: False
  project: pc_dataset_eval
  seed: 42
  save_path: cifar10_quad-graph_data-init_k128_seed42.pt
  device: cuda
Loaded cifar10: train=(47500, 3072), val=(2500, 3072), test=(10000, 3072)
Using image_shape=(3, 32, 32), ycc=lossy, input_sharing=full
Using 47500 samples to build/estimate the circuit
Circuit built with 7505 layers
Number of parameters: 67228930
Starting training for a maxi

### Random Initialization

In [2]:
!python -m scripts.evaluate_image_circuit \
    --rg quad-graph --dataset cifar10 \
    --input-sharing full --ycc lossy \
    --activation none --inner-layer cp --use-mixing-weights \
    --k 64 --seed 42 \
    --wandb --validation-steps 250 --split-seed 42 --max-epochs 100 --patience 5 \
    --save-path cifar10_quad-graph_random-init_k64_seed42.pt

  dataset: cifar10
  root: datasets
  image_shape: None
  ycc: lossy
  input_sharing: full
  rg: quad-graph
  num_bins: None
  mi_chunk_size: None
  inner_layer: cp
  k: 128
  activation: none
  weights_init: uniform
  lr: 0.01
  T_0: 1
  eta_min: 0.0001
  weight_decay: 0.0
  batch_size: 256
  max_epochs: 200
  validation_steps: 250
  patience: 5
  min_delta: 0.0
  valid_split: 0.05
  use_scheduler: False
  alpha: 5.0
  noise_scale: 2.0
  use_mixing_weights: True
  use_estimated_weights: False
  torch_compile: False
  compile_backend: inductor
  compile_mode: default
  compile_fullgraph: False
  wandb: False
  project: pc_dataset_eval
  seed: 42
  save_path: cifar10_quad-graph_random-init_k128_seed42.pt
  device: cuda
100%|█████████████████████████████████████████| 170M/170M [14:01<00:00, 203kB/s]
Loaded cifar10: train=(47500, 3072), val=(2500, 3072), test=(10000, 3072)
Using image_shape=(3, 32, 32), ycc=lossy, input_sharing=full
Using 47500 samples to build/estimate the circuit
Circui

## Region Graph: Chow-Liu Tree

The structure is learned from the training set for both initialization methods. `--num-bins 8` only affects graph construction: the input distributions retain 256 levels. `--mi-chunk-size 128` limits the number of samples processed together when computing mutual information.

### Data Initialization

In [ ]:
!python -m scripts.evaluate_image_circuit \
    --rg chow-liu-tree --dataset cifar10 \
    --input-sharing none --ycc lossy \
    --activation none --inner-layer cp --use-mixing-weights \
    --use-estimated-weights --alpha 5.0 --noise-scale 2.0 \
    --k 64 --seed 42 \
    --num-bins 8 --mi-chunk-size 128 \
    --wandb --validation-steps 250 --split-seed 42 --max-epochs 100 --patience 5 \
    --save-path cifar10_chow-liu-tree_data-init_k64_seed42.pt

### Random Initialization

In [ ]:
!python -m scripts.evaluate_image_circuit \
    --rg chow-liu-tree --dataset cifar10 \
    --input-sharing none --ycc lossy \
    --activation none --inner-layer cp --use-mixing-weights \
    --k 64 --seed 42 \
    --num-bins 8 --mi-chunk-size 128 \
    --wandb --validation-steps 250 --split-seed 42 --max-epochs 100 --patience 5 \
    --save-path cifar10_chow-liu-tree_random-init_k64_seed42.pt